# freeCAM PI-CAM interactive walkthrough

This notebook follows the clear, state-first UI style of the FreeCESM prototype while running the real iCESM CAM numerical implementation. Python owns the action order and rank-local StatePool; original Fortran kernels execute through generated adapters on 512 MPI ranks.

The main UI is deliberately small:

- `cam.state.summary()` and `cam.state.plot()` inspect real CAM fields.
- `cam.workflow` displays the current Python-owned process order.
- `cam.advance()` runs complete source-ordered steps.
- `cam.fields`, `cam.physics`, `cam.phases`, and `cam.kernels` expose fine-grained experiments.


In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
from pathlib import Path
import json
import numpy as np
import os
import subprocess

from freecam.pi_cam import (
    PICAMCase,
    PICAMNotebookSession,
    PICAMStepPlan,
)

repo = Path(os.path.expandvars('/glade/work/$USER/freeCAM'))
config_path = repo / 'configs/pi_cam_icesm131.yaml'
case = PICAMCase.from_yaml(config_path)
case.config.to_payload()


## 1. Inspect the case and workflow without starting MPI

The case configuration and default `PICAMStepPlan` can be inspected before a PBS job exists. This is the real source-faithful PI-CAM order, not a toy workflow.


In [ ]:
default_workflow = PICAMStepPlan.default()
{
    'case': case.config.case_name,
    'mpi_ranks': case.config.mpi_size,
    'steps': case.config.stop_n,
    'phases': default_workflow.phases,
    'enabled_actions': len(tuple(default_workflow)),
    'all_actions': len(default_workflow.actions),
}


In [ ]:
for action in default_workflow:
    print(f'{action.phase:10s}  {action.name:36s}  →  {action.operation}')


## 2. Run or inspect the 512-rank scientific gate

The maintained cpudev job uses one `mpiexec -n 512`, replayed rank-local coupler inputs, the fixed-address CAM `.so`, and a fail-closed comparison against the original iCESM oracle. Set `submit = True` only when a new 50-step gate is required.


In [ ]:
submit = False
job = repo / 'validation/jobs/pi_cam_python_zero_copy_state_50step.pbs'
if submit:
    submitted_job = subprocess.run(
        ['qsub', str(job)], check=True, capture_output=True, text=True
    ).stdout.strip()
    print('submitted:', submitted_job)
else:
    print('Not submitted; set submit = True to run the 512-rank gate.')


In [ ]:
evidence = repo / 'validation/pi_cam_python_zero_copy_state_vs_oracle_50step_bfb.json'
json.loads(evidence.read_text()) if evidence.exists() else {
    'status': 'Run the previous cell and wait for the PBS job to finish.'
}


## 3. Start one persistent CAM model

This creates one cpudev job and one 512-rank MPI world. Every later cell reuses the same live rank-local arrays. Only `cam.close()` ends the worker.


In [ ]:
scratch = Path(os.environ.get(
    'SCRATCH',
    os.path.expandvars('/glade/derecho/scratch/$USER'),
))
oracle_case = Path(os.path.expandvars(
    '/glade/work/$USER/CESM_cases/'
    'f.e13.F1850C5.ne16_g16.icesm131_ihesp.PI-cam-oracle.50step'
))
oracle_run = scratch / (
    'pyCAM/PI-cam/'
    'f.e13.F1850C5.ne16_g16.icesm131_ihesp.PI-cam-oracle.50step/run'
)
replay_root = scratch / (
    'pyCAM/PI-cam/nonpic-boundary-capture-50step/boundary/replay'
)
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
interactive_run = scratch / f'pyCAM/PI-cam/notebook-{stamp}/run'
interactive_run.mkdir(parents=True, exist_ok=False)
subprocess.run([
    'rsync', '-a',
    '--exclude', '*.cam.*.nc', '--exclude', '*.log.*',
    '--exclude', 'rpointer.*', '--exclude', 'timing/',
    f'{oracle_run}/', f'{interactive_run}/',
], check=True)
(interactive_run / 'timing/checkpoints').mkdir(parents=True)
interactive_run


In [ ]:
cam = PICAMNotebookSession(
    config_path,
    boundary=replay_root,
    run_dir=interactive_run,
    env_script=oracle_case / '.env_mach_specific.sh',
    python_executable=repo / '.venv/bin/python',
)
cam.start()
cam.status


## 4. Inspect the live workflow and initial atmospheric state

`cam.workflow` is a live Jupyter table. Disabled experimental leaves remain visible but dimmed. The plots retrieve only the selected rank's `T`, `u`, `v`, `q`, and `ncol`; padded `pcols` entries are excluded before computing a horizontal-mean profile.


In [ ]:
cam.workflow


In [ ]:
print('Initial state:', cam.state.summary(rank=0))
fig, axes = cam.state.plot(rank=0, label='initial')


## 5. Advance one complete CAM step and compare profiles

The orange curves are calculated from the same rank-local NumPy arrays after the original Fortran kernels finish. This is a complete CAM step, including boundary import/export, phases, clock advancement, and output actions.


In [ ]:
cam.advance(steps=1)

for axis, variable in zip(axes.flat, ('T', 'u', 'v', 'q')):
    cam.state.plot_profile(
        variable,
        rank=0,
        ax=axis,
        color='tab:orange',
        label='step 1',
    )

print('After step 1:', cam.state.summary(rank=0))
fig


## 6. Add a field and a Notebook Python process

This is the real-runtime equivalent of adding a custom `Physics` object in FreeCESM. Every rank creates its own Fortran-contiguous tracer array. `cloudpickle` sends the callback to every rank, and the process is inserted directly after `dadadj` in `cam_run1`.


In [ ]:
tracer = cam.fields.create(
    'experiment_tracer',
    dims=('pcols', 'pver', 'chunks'),
    units='kg kg-1',
    initial=0.0,
    aliases=('tracer',),
    standard_name='experiment_tracer',
)

def add_notebook_tracer(fields, context, *, rate):
    fields['tracer'][...] += rate * context.timestep_seconds

python_process = cam.physics.install_python(
    add_notebook_tracer,
    name='notebook_tracer_source',
    phase='cam_run1',
    after='dadadj',
    writes=('tracer',),
    parameters={'rate': 1.0e-6},
)

python_trace = python_process.run()
print('Tracer:', tracer.stats(rank='global'))
cam.state.plot_profile('experiment_tracer', rank=0, color='tab:green')


In [ ]:
# The workflow table updates immediately after install/move/enable/disable.
python_process.move(before='deep_convection')
python_process.disable()
python_process.enable()
cam.workflow


## 7. Build and load a Fortran process while CAM remains alive

The target rank builds a generated `bind(C)` adapter and cached `.so`; every MPI rank verifies and loads the same shared object. The new process appears in the same workflow table as native and Notebook-Python processes.


In [ ]:
runtime_temperature = cam.fields.create(
    'runtime_temperature',
    dims=('nphys_local', 'pver'),
    units='K',
    initial=240.0,
    standard_name='runtime_plugin_temperature',
)
runtime_increment = cam.fields.create(
    'runtime_temperature_increment',
    dims=(),
    units='K',
    initial=1.5,
    writable=False,
    standard_name='runtime_plugin_temperature_increment',
)
fortran_process = cam.physics.install_fortran(
    repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=repo,
    process='runtime_temperature_offset',
    phase='cam_run1',
    after='dadadj',
    unsafe=True,
)
fortran_trace = fortran_process.run()
print('Runtime Fortran temperature:', runtime_temperature.stats(rank='global'))
cam.state.plot_profile('runtime_temperature', rank=0, color='tab:red')


In [ ]:
fortran_process.move(before='deep_convection')
fortran_process.disable()
fortran_process.enable()
cam.workflow


## 8. Optional fine-grained experiments

Phase expansion replaces admitted composite stages with ordered original callees, so a routine is not executed twice. A raw kernel or one complete phase can also be called independently when its scientific preconditions already exist.


In [ ]:
expanded_cam_run1 = cam.phases.cam_run1.expand()
expanded_cam_run2 = cam.phases.cam_run2.expand()
expanded_cam_run4 = cam.phases.cam_run4.expand()
expanded_step = cam.advance(steps=1)

# Call the admitted raw-array dadadj leaf, or one complete phase, alone.
dadadj_kernel_trace = cam.kernels.dadadj.run()
cam_run3_trace = cam.phases.cam_run3.run()

{
    'cam_run1_actions': len(expanded_cam_run1),
    'cam_run2_actions': len(expanded_cam_run2),
    'cam_run4_actions': len(expanded_cam_run4),
    'step': expanded_step['step'],
    'dadadj_kernel': dadadj_kernel_trace,
    'cam_run3': cam_run3_trace,
}


## 9. Remove runtime extensions safely

A process must be removed before deleting a field that it references. Removing these UI experiments does not restart MPI or disturb the remaining native StatePool.


In [ ]:
python_process.remove()
tracer.delete()

fortran_process.remove()
runtime_temperature.delete()
runtime_increment.delete()

{
    'dynamic_fields': cam.status['dynamic_fields'],
    'python_processes': cam.status['python_processes'],
    'fortran_processes': cam.status['fortran_processes'],
}


## 10. Release the persistent MPI job

Run this cell when finished. It calls CAM finalize, exits all 512 ranks, and releases the PBS allocation.


In [ ]:
cam.close()
